# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

---

**My lane is Lane 2 — Refresh / Content Opportunity Scoring** (chosen in ML-02, framed in
ML-03). The research question:

> *Given a portfolio of existing pages and a reviewer who can look at about 50 pages a
> week, which pages should be reviewed first for a content refresh — and can a model rank
> them better than the transparent rules a human would write?*

**The decision it supports:** the weekly ordering of the refresh-review queue. The unit of
analysis is one content page (one row per content item per snapshot). The output is a
ranked queue with a suggested action and plain-language reason codes per page (ML-10 turns
it into the working playbook).

**The cost of a wrong call:** a false positive wastes one scarce review slot; a false
negative leaves a declining page — one that still has an audience — unreviewed for another
week.

**Why this is a real problem, in one measurement:** 16,262 of the 30,000 pages (54.2%)
carry the observed decline label, and those pages hold 51.3% of all 90-day impressions
(ML-02, recomputed below). At 50 reviews a week, working through just the declining pages
in portfolio order would take about 325 weeks. Ordering is not optional — the only
question is whether the order is any good. The obvious shortcut fails, too: the "just
review stale pages" rule misses the problem, because most declining pages are not old —
ML-06's audit found the "older pages decline" belief comes out **OPPOSITE** in this data.


In [1]:

# Section 1 — the size of the problem, in numbers.
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

raw = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
raw["y"] = (raw["trend_direction"] == "down").astype(int)
declining = raw[raw["y"] == 1]
print(f"Pages in portfolio:           {len(raw):,}")
print(f"Observed declining pages:     {len(declining):,} ({raw['y'].mean():.1%})")
print(f"Share of all 90d impressions: {declining['impressions_90d'].sum() / raw['impressions_90d'].sum():.1%}")
print(f"Weeks to review every declining page at 50/week: {len(declining) / 50:.0f}")
print("Ordering is not optional — the only question is whether the order is any good.")


Pages in portfolio:           30,000
Observed declining pages:     16,262 (54.2%)
Share of all 90d impressions: 51.3%
Weeks to review every declining page at 50/week: 325
Ordering is not optional — the only question is whether the order is any good.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

---

**Which release:** the starter release that ships with this repo —
`data/raw/content_refresh_anonymized.csv`, 30,000 rows × 44 columns, pseudonymized. I did
not use the gated warehouse release: the starter slice is the dataset every weekly
assignment shares, which keeps all numbers from ML-02 to ML-10 comparable, and it is the
release my validation design was built on.

**Grain (ML-04 data contract):** one row per content item per snapshot; 32 pseudonymized
clients contributing 3 to 7,008 rows each; metrics are trailing-90-day aggregates. There
are **no calendar dates** in the file, so no time split and no seasonality claim is
possible — I state that plainly rather than faking a time axis (ML-08, Section 2).

**Excluded on purpose:**

- `trend_direction` and `trend_pct` — they *define* the label (Section 3), so they are
  banned as features everywhere in this project (ML-05).
- `content_id` / `client_id` as features — pseudonyms, used for grouping, joining and
  splitting only (DATA_USE.md).
- Product decision flags — not shipped in this release at all, so nothing here learns the
  product's own decisions.

**Missingness is patterned, not random** (ML-04): it follows `content_type`. Step 01 of
the pipeline therefore adds availability flags (`has_clicks`, `has_ai_sessions`,
`measurable_opportunity`) instead of blind-filling — a blind `fillna(0)` would inject a
category signal. Two readings to keep straight: `avg_position = 0` means *no data*
(1,205 pages), and rate columns are percentages on a 0–100 scale (`ctr = 0.76` means
0.76%).

**Public safety:** this release contains no client names, URLs, page titles or raw
queries. Every ID printed in this notebook is a shipped pseudonym.


In [2]:

# Section 2 — the data contract in one screen (ML-04 has the full version).
print(f"Shape: {raw.shape[0]:,} rows x {raw.shape[1] - 1} shipped columns (+1 label column derived in Section 1)")
sizes = raw.groupby("client_id").size()
print(f"Clients: {raw['client_id'].nunique()} pseudonymized, {sizes.min()}–{sizes.max()} pages each")
print(f"avg_position = 0 means NO DATA: {(raw['avg_position'] == 0).sum():,} pages")
miss = raw.isna().mean().sort_values(ascending=False)
print("\nMost-missing columns (missingness follows content_type — patterned, not random):")
print((miss[miss > 0] * 100).round(1).head(6).to_string())
date_like = [c for c in raw.columns if c.endswith("_date") or c == "date"]
print(f"\nCalendar-date columns: {date_like or 'none'} — trailing-90-day aggregates only, "
      "so no time split is possible.")
print("Banned as features (label-derived): trend_direction, trend_pct")
print("Grouping/splitting only (pseudonyms): content_id, client_id")


Shape: 30,000 rows x 44 shipped columns (+1 label column derived in Section 1)
Clients: 32 pseudonymized, 3–7008 pages each
avg_position = 0 means NO DATA: 1,205 pages

Most-missing columns (missingness follows content_type — patterned, not random):
provider_used      71.5
word_count         25.7
char_count         25.7
char_count_tier    25.7
word_count_tier    25.7
model_used         19.1

Calendar-date columns: none — trailing-90-day aggregates only, so no time split is possible.
Banned as features (label-derived): trend_direction, trend_pct
Grouping/splitting only (pseudonyms): content_id, client_id


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

---

**Label.** `is_declining_label = 1` exactly when `trend_direction == "down"` — an
*observed* decline label derived from the trailing-window comparison (`trend_pct`). That
derivation is precisely why both parent columns are banned as features: a model fed
`trend_pct` would be reading the answer key (ML-05 demonstrates it scoring a perfect,
meaningless 1.000).

**Features.** The reference pipeline's list, kept identical so every comparison stays
clean: 18 numeric features (`search_volume`, `competition`, `cpc`, `word_count`,
`char_count`, the log-transformed 90-day traffic counts, `days_with_impressions`,
`days_with_sessions`, `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`,
`engagement_rate`, `scroll_rate`, `ai_traffic_pct`) plus 8 categorical features
one-hot encoded (`competition_level`, `content_type`, `main_intent`, `age_tier`,
`freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`). IDs never enter
the matrix. The code below asserts the banned columns are absent.

**Baselines.** Two transparent hand rules, scored on the *same* holdout: the reference
pipeline's rule (P@50 = 0.24) and my own points rule from ML-07 (P@50 = 0.48) — plus a
dummy floor that sits at the base rate, so every real number has something honest under
it.

**Validation design.** Client holdout, seed 42: 26 clients train (27,675 pages), 6 clients
test (2,325 pages), zero client overlap — asserted in code. Pages from one client share
site-level character, so a random row split would let the model memorise *which client* a
page belongs to; ML-09 measured that inflation directly (same forest: P@50 0.90 random
split vs 0.68 client holdout). The deployment question is "does the ranking work on a
portfolio I haven't modelled?", and only the grouped split asks it. **Metrics:**
Precision@K (K = 20/50/100) — the metric that describes an ordered queue — with ROC AUC
and average precision alongside. The holdout base rate is **0.391** (full-data rate
0.542); it is printed next to every P@K, because a precision number without its base rate
is only half a claim.

**Models.** Dummy (prior), logistic regression (scaled, class-weighted), decision tree
(depth 5), random forest (200 trees, depth 10, class-weighted) — the pipeline's own
learners, on purpose. **One experiment on top:** a second forest adding the three
availability flags step 01 engineers but the reference list leaves out.


In [3]:

# Section 3 — feature matrix + the honest split. Mirrors scripts/03_train_model.py
# (read-only use — scripts/ stays pristine, per GUIDE).
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
fv = pd.read_csv(ROOT / "data" / "processed" / "refresh_feature_vector.csv")
BANNED = {"trend_direction", "trend_pct", "content_id", "client_id", "is_declining_label"}

NUMERIC = ["search_volume", "competition", "cpc", "word_count", "char_count",
           "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
           "days_with_impressions", "days_with_sessions", "content_age_days",
           "days_since_last_update", "ctr", "avg_position", "engagement_rate",
           "scroll_rate", "ai_traffic_pct"]
CATEGORICAL = ["competition_level", "content_type", "main_intent", "age_tier",
               "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]
assert not (set(NUMERIC) | set(CATEGORICAL)) & BANNED, "banned column in feature list"

def build_matrix(frame, extra_numeric=()):
    num = frame[NUMERIC + list(extra_numeric)].apply(pd.to_numeric, errors="coerce")
    num = num.replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = pd.get_dummies(frame[CATEGORICAL].fillna("unknown").astype(str),
                         prefix=CATEGORICAL, dummy_na=False, dtype=float)
    return pd.concat([num.reset_index(drop=True), cat.reset_index(drop=True)], axis=1)

X = build_matrix(fv)
X_flags = build_matrix(fv, extra_numeric=["has_clicks", "has_ai_sessions", "measurable_opportunity"])
y = fv["is_declining_label"].astype(int)

clients = fv["client_id"].fillna("unknown").astype(str)
unique_clients = clients.drop_duplicates().to_numpy()
shuffled = np.random.default_rng(SEED).permutation(unique_clients)
test_clients = set(shuffled[:max(1, int(round(len(shuffled) * 0.2)))])
test_mask = clients.isin(test_clients).to_numpy()
train_idx, test_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
assert not set(clients.iloc[train_idx]) & set(clients.iloc[test_idx]), "client overlap!"
print(f"Client holdout (seed {SEED}): {len(train_idx):,} train pages / {len(test_idx):,} test pages, "
      "zero client overlap")
print(f"Base rates — full data {y.mean():.3f} | train {y.iloc[train_idx].mean():.3f} | "
      f"holdout {y.iloc[test_idx].mean():.3f}")


Client holdout (seed 42): 27,675 train pages / 2,325 test pages, zero client overlap
Base rates — full data 0.542 | train 0.555 | holdout 0.391


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

---

**The honest table** (client holdout, seed 42; recomputed live below — every row
matches the ML-08 receipts in `work/outputs/w05_model_metrics.json` exactly, except the
flags experiment, which lands within a page or two at each K: boundary sensitivity in a
variant whose top 50 is one client's pages, noted again under the table):

| Model | P@20 | P@50 | P@100 | ROC AUC | Avg precision |
|---|---:|---:|---:|---:|---:|
| reference rule (scripts/02) | 0.15 | 0.24 | 0.36 | 0.627 | 0.468 |
| my points rule (ML-07) | 0.50 | 0.48 | 0.44 | 0.573 | 0.440 |
| dummy (prior) | 0.30 | 0.40 | 0.37 | 0.500 | 0.391 |
| logistic regression | 0.35 | 0.40 | 0.44 | 0.700 | 0.522 |
| decision tree | 0.65 | 0.66 | 0.68 | 0.742 | 0.575 |
| **random forest** | **0.70** | **0.68** | **0.70** | **0.747** | **0.610** |
| random forest + has_ flags (experiment) | 0.90 | 0.82 | 0.78 | 0.749 | 0.628 |

(Holdout base rate 0.391 — random ordering scores ≈0.39 at any K. The dummy's exact top-K
wobbles with tie-ordering; its role is the floor, not a contender.)

**The headline claim, at the level the evidence carries:** on clients the model had never
seen, the random forest's top 50 contained about **2.8× as many observed-declining pages
as the reference rule's top 50** (P@50 0.68 vs 0.24), and 1.4× my own hand rule (0.48).
That is a *ranking* claim about review order — measured, out-of-sample, decision-support.
Nothing here claims a refresh *causes* recovery; this data cannot say that.

**The flags variant is the biggest number in the table and I do not headline it.** Its
top 50 is 49 pages from a single held-out client (ML-09; measured again in Section 5), so
its P@50 (0.84 in ML-08, 0.82 on this rerun) may be one portfolio's character rather
than a general improvement. It stays in
the table as a directional result that needs a wider holdout before it earns a claim.

**What the model leans on:** visibility and age — `log_impressions_90d`,
`days_with_impressions`, `avg_position`, `content_age_days` lead the importances (chart in
Section 7). Reassuring: the model found the reviewer's intuition, weighted better. No
feature is suspiciously perfect, and no ID is in the matrix.

**Error shape (ML-08).** False positives are substantial pages that *held up* — they look
like declining pages on every observable I have; that is the honest ceiling of this
feature set. False negatives are near-invisible pages (1–3 impressions, position
unmeasured): a page nobody sees cannot show a measurable decline pattern. The queue is a
review-priority tool for pages with an audience — never evidence that quiet pages are
healthy.

**The deliverable.** Blended with the baseline and exported by the pipeline, the final
queue over all 30,000 pages carries 3,576 high-confidence, 11,424 medium and 15,000
low-confidence pages; actions: `monitor` 13,069 · `refresh` 8,207 ·
`refresh_and_review_ctr` 6,655 · `refresh_and_review_engagement` 1,987 ·
`expand_and_refresh` 82. ML-10 turns it into the weekly playbook.


In [4]:

# Section 4 — the honest table, recomputed in one run on the Section-3 holdout.
from sklearn.metrics import average_precision_score, roc_auc_score

def precision_at_k(y_true, scores, k):
    # Copied verbatim from scripts/ml_utils.py — the pipeline's own helper, so this
    # recomputation follows the exact path that produced the ML-08 receipts
    # (including how it orders tied scores).
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0

def score_row(name, scores):
    yt = y.iloc[test_idx].to_numpy()
    scores = np.asarray(scores, dtype=float)
    return {"model": name,
            "P@20": round(precision_at_k(yt, scores, 20), 2),
            "P@50": round(precision_at_k(yt, scores, 50), 2),
            "P@100": round(precision_at_k(yt, scores, 100), 2),
            "ROC AUC": round(float(roc_auc_score(yt, scores)), 3),
            "Avg precision": round(float(average_precision_score(yt, scores)), 3)}

rows = []
base_q = pd.read_csv(ROOT / "data" / "processed" / "baseline_refresh_queue.csv")
base_score = dict(zip(base_q["content_id"], base_q["baseline_refresh_score"]))
rows.append(score_row("reference_rule (scripts/02)", fv["content_id"].map(base_score).iloc[test_idx]))
my_q = pd.read_csv(ROOT / "work" / "outputs" / "baseline_action_score.csv")
my_score = dict(zip(my_q["content_id"], my_q["my_score"]))
rows.append(score_row("my_rule (ML-07 points)", fv["content_id"].map(my_score).iloc[test_idx]))

models = {
    "dummy (prior)": DummyClassifier(strategy="prior", random_state=SEED),
    "logistic_regression": Pipeline([("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED))]),
    "decision_tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5,
                                            min_samples_leaf=50, random_state=SEED),
    "random_forest": RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                                            min_samples_leaf=25, n_estimators=200, n_jobs=-1,
                                            random_state=SEED),
}
fitted = {}
for name, model in models.items():
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    fitted[name] = model
    rows.append(score_row(name, model.predict_proba(X.iloc[test_idx])[:, 1]))

rf_flags = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                                  min_samples_leaf=25, n_estimators=200, n_jobs=-1,
                                  random_state=SEED)
rf_flags.fit(X_flags.iloc[train_idx], y.iloc[train_idx])
flags_scores = rf_flags.predict_proba(X_flags.iloc[test_idx])[:, 1]
rows.append(score_row("random_forest + has_ flags (experiment)", flags_scores))

results = pd.DataFrame(rows)
print(f"Holdout base rate: {y.iloc[test_idx].mean():.3f} — random ranking scores ~this at any K.")
print(results.to_string(index=False))
print("\nReceipts: work/outputs/w05_model_metrics.json (ML-08), "
      "work/outputs/w06_validation_audit.json (ML-09).")
print("Every row matches the ML-08 receipts exactly except the flags experiment "
      "(P@50 0.82 here vs 0.84 there) —")
print("a one-page boundary difference in the variant whose top 50 is a single client's pages.")


Holdout base rate: 0.391 — random ranking scores ~this at any K.
                                  model  P@20  P@50  P@100  ROC AUC  Avg precision
            reference_rule (scripts/02)  0.15  0.24   0.36    0.627          0.468
                 my_rule (ML-07 points)  0.50  0.48   0.44    0.573          0.440
                          dummy (prior)  0.30  0.40   0.37    0.500          0.391
                    logistic_regression  0.35  0.40   0.44    0.700          0.522
                          decision_tree  0.65  0.66   0.68    0.742          0.575
                          random_forest  0.70  0.68   0.70    0.747          0.610
random_forest + has_ flags (experiment)  0.90  0.82   0.78    0.749          0.628

Receipts: work/outputs/w05_model_metrics.json (ML-08), work/outputs/w06_validation_audit.json (ML-09).
Every row matches the ML-08 receipts exactly except the flags experiment (P@50 0.82 here vs 0.84 there) —
a one-page boundary difference in the variant whose top 50 is

## 5. Limitations

*What this work cannot claim.*

---

What this work cannot claim, stated before a reader has to:

- **One snapshot, cross-sectional.** I measured association and ranking quality — never
  refresh impact. Whether acting on the queue improves a page's trajectory needs an
  experiment (refresh a random subset, hold the rest) that this data cannot run. Every
  claim in this notebook is decision-support, none is causal.
- **No calendar time.** Trailing-90-day aggregates only: no seasonality, no past→future
  validation. The client holdout is the strongest honest split this file supports, and it
  is not a time split.
- **Slice-dependent metrics.** The holdout base rate (0.391) is lower than the full-data
  rate (0.542) because the six held-out clients decline less. P@K travels with the slice;
  AUC and lift-over-baseline are the more portable numbers.
- **Measurement floors and gaps.** 12.5% of pages have under 10 impressions in 90 days
  (measured below) — they cannot be ranked meaningfully. `avg_position = 0` is
  missing-not-zero on 1,205 pages; position information was only partly measured in this
  snapshot (ML-06); `days_since_last_update` is coarse (ML-07).
- **Concentration risk.** Top-K lists can be captured by one large client — 15/20 in my
  ML-07 rule's top 20, and 49/50 in the flags variant's holdout top 50 (measured below).
  The playbook caps per-client share in a weekly batch, but the underlying ranking still
  reflects portfolio mix.
- **Version sensitivity.** The GUIDE's FAQ notes the forest's P@50 reproduces at
  0.68–0.74 depending on library versions. The stable claim is the ~3× lift over the
  reference rule, not the third decimal.
- **Scope.** 32 pseudonymized clients, one snapshot, one portfolio family. Directional
  for similar portfolios; not a claim about "the algorithm" or about search in general.
- **A negative result I kept:** my signal audit's "older pages decline" belief came out
  OPPOSITE (ML-06), and logistic regression did not beat my hand rule. Both are in the
  record, because a playbook built only on confirming results would not deserve trust.


In [5]:

# Section 5 — two limitations, measured rather than asserted.
quiet = fv["impressions_90d"] < 10
print(f"Measurement floor: {quiet.mean():.1%} of pages have <10 impressions in 90 days; their observed")
print(f"decline rate is {fv.loc[quiet, 'is_declining_label'].mean():.3f} vs "
      f"{fv.loc[~quiet, 'is_declining_label'].mean():.3f} for pages with an audience.")

_tmp = pd.DataFrame({"score": flags_scores}).reset_index()
order = _tmp.sort_values("score", ascending=False).head(50)["index"].to_numpy()
top50_clients = clients.iloc[test_idx].to_numpy()[order]
mix = pd.Series(top50_clients).value_counts()
print(f"\nConcentration: the flags variant's holdout top 50 draws {mix.iloc[0]}/50 pages from a single")
flags_p50 = float(results.loc[results["model"].str.contains("flags"), "P@50"].iloc[0])
print(f"client ({mix.index[0]}). That is why its P@50 = {flags_p50:.2f} stays a directional result,")
print("not the headline — the headline belongs to the plain forest's 0.68.")


Measurement floor: 12.5% of pages have <10 impressions in 90 days; their observed
decline rate is 0.201 vs 0.591 for pages with an audience.

Concentration: the flags variant's holdout top 50 draws 49/50 pages from a single
client (client_f74efabef1). That is why its P@50 = 0.82 stays a directional result,
not the headline — the headline belongs to the plain forest's 0.68.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

---

The playbook output (ML-10), condensed — this is the paper's recommendations section.
Ranked by what a reviewer should do first:

1. **Review the blended queue top-down in weekly batches of 50, high-confidence tier
   first.** At the measured holdout rate, a model-ordered batch of 50 holds roughly 34
   pages warranting action, against ~12 for the reference rule's ordering.
2. **Cap any single client at 20 pages per weekly batch** and fill from the next clients
   down the queue — the guard against the concentration measured in Section 5.
3. **Run the measurement check before any edit:** high impressions with exactly-zero
   clicks or engagement means *verify tracking first* (ML-07 pick #3; ML-10 flags it
   automatically on the top 100).
4. **Match the action to the reason codes:** `refresh_and_review_ctr` pages get
   snippet/intent review before deep rewrites; `expand_and_refresh` (thin but seen) gets
   depth work; the `monitor` tier gets no edits.
5. **Keep the transparent rule as fallback and cross-check.** Where the model and my
   points rule agree, confidence is highest; where they disagree, the reason codes say
   why, in words.
6. **Never automate** publishing, deletion or redirects from the queue, and never present
   the queue to a client as a recovery promise (ML-10's no-go list).
7. **Re-validate on every new snapshot** against the ML-10 fingerprint; replace the model
   only when a fresh client holdout matches or beats P@50 = 0.68.


In [6]:

# Section 6 — the playbook's first page: this week's batch shape (ML-10 has the full playbook).
queue = pd.read_csv(ROOT / "outputs" / "refresh_queue.csv")
playbook_path = ROOT / "work" / "outputs" / "playbook_queue_top100.csv"
if playbook_path.exists():
    top = pd.read_csv(playbook_path)
    print("Source: ML-10 export (work/outputs/playbook_queue_top100.csv)")
else:
    top = queue.head(100)
    print("Source: reference pipeline queue (outputs/refresh_queue.csv)")
print(top.head(10)[["final_rank", "final_refresh_score", "confidence", "suggested_action"]].to_string(index=False))

picked, counts = [], {}
for idx, row in queue.iterrows():
    c = row["client_id"]
    if counts.get(c, 0) >= 20:
        continue
    picked.append(idx)
    counts[c] = counts.get(c, 0) + 1
    if len(picked) == 50:
        break
print(f"\nWeek-1 batch: 50 pages with a per-client cap of 20 spans {len(counts)} clients; "
      f"the uncapped top 50 spans {queue.head(50)['client_id'].nunique()} clients "
      f"with {queue.head(50)['client_id'].value_counts().iloc[0]} pages from one client.")


Source: ML-10 export (work/outputs/playbook_queue_top100.csv)
 final_rank  final_refresh_score confidence       suggested_action
          1            81.928467       high refresh_and_review_ctr
          2            81.728449       high refresh_and_review_ctr
          3            81.639118     medium refresh_and_review_ctr
          4            80.804986     medium refresh_and_review_ctr
          5            80.801530       high refresh_and_review_ctr
          6            80.752578       high refresh_and_review_ctr
          7            80.632372       high refresh_and_review_ctr
          8            80.439403     medium                refresh
          9            80.204325       high refresh_and_review_ctr
         10            80.146808       high refresh_and_review_ctr

Week-1 batch: 50 pages with a per-client cap of 20 spans 4 clients; the uncapped top 50 spans 4 clients with 36 pages from one client.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

---

The code below generates the four charts the deployed paper embeds, saved to
`work/figures/`:

1. **`capstone_p50_comparison.png`** — P@50 for every contender, rules first: the honest
   comparison in one glance.
2. **`capstone_precision_at_k.png`** — Precision@K curves for the two rules and the
   forest, against the holdout base rate.
3. **`capstone_action_mix.png`** — the final queue by suggested action (what the
   recommendations in Section 6 actually distribute).
4. **`capstone_feature_importance.png`** — the forest's top 10 features.

Plus the tables and receipts the paper quotes: the honest table printed in Section 4;
`work/outputs/w05_model_metrics.json` and `work/outputs/w06_validation_audit.json`
(ML-08/ML-09 receipts); `work/outputs/playbook_summary.json` and
`work/outputs/playbook_action_mix.png` (ML-10). The companion write-up is
`work/capstone_report.md`; the deployed paper's URL goes in `submission/paper_url.txt`
at submission time.


In [7]:

# Section 7 — artifacts the deployed paper embeds (saved to work/figures/).
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

figdir = ROOT / "work" / "figures"
figdir.mkdir(parents=True, exist_ok=True)

# 1) P@50: rules vs models — the honest comparison
plot = results.set_index("model")["P@50"].iloc[::-1]
fig, ax = plt.subplots(figsize=(6.5, 3.4))
plot.plot.barh(ax=ax)
ax.set_xlabel("Precision@50 on client holdout (base rate 0.391)")
ax.set_title("Rules vs models — same split, same metric")
fig.tight_layout()
fig.savefig(figdir / "capstone_p50_comparison.png", dpi=150)
plt.close(fig)

# 2) Precision@K curves
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ks = [10, 20, 50, 100, 200, 500]
series = {"reference_rule": fv["content_id"].map(base_score).iloc[test_idx],
          "my_rule": fv["content_id"].map(my_score).iloc[test_idx],
          "random_forest": fitted["random_forest"].predict_proba(X.iloc[test_idx])[:, 1]}
for name, sc in series.items():
    ax.plot(ks, [precision_at_k(y.iloc[test_idx], sc, k) for k in ks], marker="o", label=name)
ax.axhline(y.iloc[test_idx].mean(), ls="--", color="grey", label="holdout base rate")
ax.set_xlabel("K")
ax.set_ylabel("Precision@K")
ax.legend()
ax.set_title("Precision@K on the client holdout")
fig.tight_layout()
fig.savefig(figdir / "capstone_precision_at_k.png", dpi=150)
plt.close(fig)

# 3) Final queue action mix
fig, ax = plt.subplots(figsize=(6.5, 3.2))
queue["suggested_action"].value_counts().plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_xlabel("pages")
ax.set_title("Final blended queue by suggested action (30,000 pages)")
fig.tight_layout()
fig.savefig(figdir / "capstone_action_mix.png", dpi=150)
plt.close(fig)

# 4) Random-forest feature importances (top 10)
imp = pd.Series(fitted["random_forest"].feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
imp.iloc[::-1].plot.barh(ax=ax)
ax.set_xlabel("importance")
ax.set_title("Random forest — top 10 features")
fig.tight_layout()
fig.savefig(figdir / "capstone_feature_importance.png", dpi=150)
plt.close(fig)

print("Saved to work/figures/: capstone_p50_comparison.png, capstone_precision_at_k.png,")
print("capstone_action_mix.png, capstone_feature_importance.png")
print("Tables: the honest table in Section 4. Receipts: work/outputs/w05_model_metrics.json,")
print("w06_validation_audit.json, playbook_summary.json.")


Saved to work/figures/: capstone_p50_comparison.png, capstone_precision_at_k.png,
capstone_action_mix.png, capstone_feature_importance.png
Tables: the honest table in Section 4. Receipts: work/outputs/w05_model_metrics.json,
w06_validation_audit.json, playbook_summary.json.


## 8. ML-12 — 5-minute demo outline

- **0:00–0:40 — The problem in one picture.** 30,000 pages, 54.2% observed declining,
  holding 51.3% of impressions — and a reviewer who can look at ~50 pages a week.
  Ordering is the whole job.
- **0:40–1:30 — The data and the trap.** One snapshot, 32 pseudonymized clients, no
  calendar dates. The label comes from `trend_direction` — show the leak: add `trend_pct`
  as a feature and the score becomes a perfect 1.000. That is why it is banned.
- **1:30–2:30 — The honest split.** Client holdout, seed 42. Same forest, two numbers:
  random row split says P@50 0.90, grouped split says 0.68. Only one of them answers the
  deployment question.
- **2:30–3:40 — The result.** The honest table: reference rule 0.24 → my hand rule 0.48 →
  random forest 0.68 (P@50, unseen clients). Then the flags variant (0.84 in ML-08, 0.82 on the capstone rerun) — with its
  49-of-50-from-one-client caveat, live.
- **3:40–4:30 — The playbook.** Weekly batch of 50, per-client cap of 20, measurement
  check before any edit, and the no-go list: nothing auto-published, nothing deleted,
  nothing promised.
- **4:30–5:00 — Limits and close.** One snapshot, cross-sectional, no causal claim. The
  queue is decision-support for review order. Repo + paper link for anyone who wants to
  re-run it.


## 9. ML-12 — Social-post cut

> 54% of the pages in a 30,000-page portfolio were quietly declining — and they held half
> the audience. I built a refresh-review queue that ranks which pages an editor should
> look at first: on clients the model had never seen, its top 50 held ~2.8× as many
> declining pages as the transparent rule's top 50 (Precision@50 0.68 vs 0.24). The most
> important chart isn't the score, though — it's the leak: one banned column scores a
> perfect 1.000, because it *defines* the label. Honest validation is the whole game.
> [Chart: `work/figures/capstone_p50_comparison.png`] Full write-up + notebooks: <paper
> link goes here at submission>


## 10. ML-12 — Employer-facing summary (3 sentences)

I built an end-to-end ML ranking system that prioritises content-refresh review across a
30,000-page, 32-client anonymized search portfolio, improving a transparent hand rule
(Precision@50 0.24) into a validated random-forest queue (0.68) under a strict
client-holdout design. The work included a data contract, a leakage audit (a
label-derived column scores a perfect 1.000 and is banned), a signal audit with
confirmed/opposite/mixed verdicts, and a reviewer playbook with monitoring and retrain
triggers. All claims are decision-support framed, and everything reproduces from the
notebooks in this repo (seed 42).


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.

*Two boxes left unticked on purpose: this notebook and `work/capstone_report.md` are complete and executed in `work/`, and the report carries all 9 paper sections — Abstract at the top, Acknowledgments & data credit (with the https://flyrank.ai link) at the bottom. Committing to my repo and deploying the paper page are my own submission steps, handled separately.*
